# Session 2 — Random Variables

**Goal:** stop treating `age`, `chol`, and `thalach` as columns of numbers and start
treating each as a **random variable** with an expectation, a variance, and a shape —
the summary that every later session compresses a column down to before using it.

## What this stage does for the system

Session 1 asked questions about *events* ("does this patient have disease?"). This
session asks about *quantities* ("how much cholesterol, and how much does it vary?"),
because the inputs feeding the screening system are quantities, not events. Two
numbers — expectation and variance — are what a model, a t-test, a confidence
interval, and a standardisation step all reduce a column to before doing anything
else. If those two numbers are computed on the wrong scale, or with the wrong
denominator, or on a column whose shape makes the mean meaningless, the error
propagates silently into every one of them.

The specific trap this session defuses: **the mean is not always a summary.** For a
lopsided column, reporting a mean and standard deviation describes a distribution the
data does not have — and Session 3 picks that thread up directly.

## The dataset

Every session in this module works on one registry: the UCI **Heart Disease**
dataset (Cleveland), fetched live from the UCI ML Repository with `ucimlrepo` so the
notebooks are runnable by anyone without a CSV sitting on their machine. It holds 303
patients with clinical measurements (`age`, `trestbps` resting blood pressure, `chol`
serum cholesterol, `thalach` max heart rate achieved, `oldpeak` ST depression),
categorical findings (`sex`, `cp` chest-pain type, `fbs` fasting blood sugar > 120,
`restecg`, `exang` exercise-induced angina, `slope`, `ca`, `thal`), and the outcome
`num` — angiographic disease severity 0-4, which this module binarises into
`target` (0 = no disease, 1 = disease present).

Deliberately one dataset throughout: switching datasets between topics would mean
re-learning the data every session instead of building cumulative familiarity with
one problem, the way a real analyst does.

## How to read this notebook

Every code cell is followed by a short **Observe / Infer** note: *Observe* points at
exactly what to look at in that cell's output, and *Infer* explains what conclusion to
draw from it — and what a different result would imply. Read them before running the
next cell; several of them flag things worth double-checking before you move on.

## Prerequisites

This session runs entirely locally — no account or credentials needed.

```bash
pip install ucimlrepo pandas numpy scipy scikit-learn statsmodels matplotlib seaborn
```

## Step 1 — Load the registry from the UCI repository

Fetching directly from the UCI ML Repository keeps this notebook runnable by anyone,
instead of depending on a CSV already sitting on your machine. The same nine lines
open every session in this module, so the 297 patients below are the identical 297
patients every other notebook analyses.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

heart_disease = fetch_ucirepo(id=45)
df = pd.concat([heart_disease.data.features, heart_disease.data.targets], axis=1)

# `num` is severity 0-4; this module screens for disease presence, so binarise it.
df = df.dropna().reset_index(drop=True)
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns="num")

print(f"{len(df)} patients, {len(df.columns)} columns")
print(f"disease prevalence: {df['target'].mean():.3f}")
df.head()

**Observe:** `297 patients, 14 columns` and `disease prevalence: 0.461`. The preview
shows `age`, `sex`, `cp`, `trestbps`, `chol`, `fbs`, `restecg`, `thalach`, `exang`,
`oldpeak`, `slope`, `ca`, `thal`, and the `target` column just derived.
**Infer:** 303 rows are fetched and 297 survive `dropna()` — six patients are missing
`ca` (number of major vessels seen on fluoroscopy) or `thal`. Dropping six rows out of
303 is defensible here and keeps every notebook in this module working on the identical
297 patients; on a larger fraction of missing values you would have to impute instead,
and *that* choice would itself need the distribution work of Session 3. If your row
count is not 297, you are on a different subset than every number quoted below.

## Step 2 — Sort the inputs into discrete and continuous

A random variable is *discrete* if its outcomes are countable (disease yes/no,
chest-pain type 1-4) and *continuous* if it can take any value in a range
(cholesterol). The distinction decides which tools apply: Session 8's chi-square test
works on discrete inputs, Session 7's t-test on continuous ones. In a raw pandas
frame both arrive as `int64`, so the dtype cannot tell you which is which.

In [ ]:
inventory = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "distinct_values": df.nunique(),
    "min": df.min(numeric_only=True),
    "max": df.max(numeric_only=True),
})
print(inventory)

**Observe:** `chol` has 152 distinct values across 126-564, while `cp` has 4 and
`restecg` has 3 — yet all three are stored as `int64`.
**Infer:** the dtype is a storage detail; the *meaning* is what decides the treatment.
`cp = 4` is not "twice as much chest pain as `cp = 2`", it is a different category
that happens to be labelled with a numeral, and averaging it produces the meaningless
`3.16` you would get from averaging postcodes. The columns to watch are `cp`,
`restecg`, `slope`, and `thal` — all categorical, all numerically coded, and all of
which will need one-hot encoding rather than being fed to a regression as-is. `ca`
(0-3 major vessels) is the genuinely ambiguous one: it is a *count*, so it is ordered
and arithmetic on it is defensible, which is why Session 5 treats it as numeric.

## Step 3 — Expectation and variance

Expectation $E[X]$ is the long-run average; variance $\text{Var}(X) = E[(X-E[X])^2]$
is the average squared distance from it, and the standard deviation is its square
root — back in the original units, which is why it is the one you report.

In [ ]:
continuous = ["age", "trestbps", "chol", "thalach", "oldpeak"]

summary = df[continuous].agg(["mean", "std", "min", "max"]).T
summary["range"] = summary["max"] - summary["min"]
print(summary.round(2))

**Observe:** `chol` has mean `247.35` and std `52.00`; `thalach` mean `149.60`, std
`22.94`; `oldpeak` mean `1.06` with std `1.17` — a standard deviation larger than its
own mean.
**Infer:** the standard deviations are not comparable across rows of this table
because the units differ (mg/dL, beats/min, mm of ST depression) — 52 is not "bigger
variability" than 22.94 in any meaningful sense. Step 6 fixes that with
standardisation. The row worth stopping on is `oldpeak`: a standard deviation exceeding
the mean on a strictly non-negative quantity is only possible if the distribution is
piled up at zero with a long tail to the right, which means `mean ± std` reaches into
negative values that cannot occur. That is the first concrete case in this module of a
summary describing a distribution the data does not have.

## Step 4 — `ddof`: the denominator that quietly changes your answer

Dividing the summed squared deviations by $n$ gives the variance *of the data you
have*. Dividing by $n-1$ gives an unbiased estimate of the variance of the
*population the data was drawn from* — which is almost always the quantity you
actually want, since the registry is a sample of patients, not the whole world.

In [ ]:
import numpy as np

manual_n = ((df["chol"] - df["chol"].mean()) ** 2).mean()          # divides by n
print(f"variance, divide by n     : {manual_n:.2f}")
print(f"pandas .var(ddof=0)        : {df['chol'].var(ddof=0):.2f}   <- same thing")
print(f"pandas .var()   [ddof=1]   : {df['chol'].var():.2f}   <- pandas default")
print(f"numpy  .var()   [ddof=0]   : {np.var(df['chol']):.2f}   <- numpy default")
print()
print(f"difference: {df['chol'].var() - df['chol'].var(ddof=0):.2f}  "
      f"({100 * (df['chol'].var() / df['chol'].var(ddof=0) - 1):.2f}%)")

**Observe:** `2703.75` (ddof=1) versus `2694.65` (ddof=0) — a 0.34% difference — and
note that **pandas and numpy default to opposite conventions**.
**Infer:** at n=297 the gap is cosmetic, and that is exactly what makes it dangerous:
it is too small to notice as a bug and it never goes away. On a subgroup of 20 patients
the same discrepancy is 5%, and it lands inside t-statistics and confidence intervals
where a 5% error in the denominator moves a p-value across the 0.05 line. The habit
worth forming is to pass `ddof` explicitly whenever a variance feeds a test rather than
a report — Session 7 does exactly that when computing Welch's t-test by hand.

## Step 5 — Conditional expectation: the same idea as Session 1, applied to a mean

$E[X \mid Y]$ is Session 1's conditional probability with a mean in place of a rate:
recompute the average on the subset that shares a condition. Splitting each input by
disease status is the crudest possible feature-screening pass, and it previews exactly
what Session 7's t-test formalises.

In [ ]:
by_target = df.groupby("target")[continuous].agg(["mean", "std"])
by_target.index = ["no disease", "disease"]
print(by_target.round(2).to_string())

print("\nGap between groups, in units of the overall standard deviation:")
for col in continuous:
    gap = df.loc[df["target"] == 1, col].mean() - df.loc[df["target"] == 0, col].mean()
    print(f"  {col:9} {gap:+8.2f}   ({gap / df[col].std():+.2f} sd)")

**Observe:** `thalach` drops from `158.58` to `139.11` (−0.85 sd) and `oldpeak` rises
from `0.60` to `1.59` (+0.85 sd), while `chol` moves only `243.49 → 251.85`
(+0.16 sd).
**Infer:** expressing each gap in standard deviations is what makes the five inputs
comparable, and it immediately ranks them: `thalach` and `oldpeak` separate the groups
five times more sharply than `chol` does. That ranking survives into Session 5's
correlations and Session 9's regression coefficients, so it is worth remembering now.
Two limits on what this table can tell you. It says nothing about whether a gap is
distinguishable from chance — that is Session 6. And it treats each input in
isolation, so it cannot see that `thalach` and `oldpeak` might be measuring the same
underlying thing twice; Session 5's correlation matrix is what catches that.

## Step 6 — Linearity of expectation

$E[aX + bY] = a\,E[X] + b\,E[Y]$, and — unlike the corresponding rule for variance —
this holds whether or not $X$ and $Y$ are independent. It is what lets you reason
about the mean of a composite risk score without knowing anything about how its
components relate.

In [ ]:
# An illustrative composite score: 0.02 * cholesterol + 0.05 * resting blood pressure
a, b = 0.02, 0.05

predicted_mean = a * df["chol"].mean() + b * df["trestbps"].mean()
actual_mean = (a * df["chol"] + b * df["trestbps"]).mean()
print(f"a*E[chol] + b*E[trestbps] = {predicted_mean:.4f}")
print(f"E[a*chol + b*trestbps]    = {actual_mean:.4f}   <- identical, always")

# Variance is NOT linear: the covariance term does not vanish unless they are independent.
var_sum_naive = a**2 * df["chol"].var() + b**2 * df["trestbps"].var()
var_sum_actual = (a * df["chol"] + b * df["trestbps"]).var()
print()
print(f"a^2*Var(chol) + b^2*Var(bp) = {var_sum_naive:.4f}   <- assumes independence")
print(f"Var(a*chol + b*trestbps)    = {var_sum_actual:.4f}   <- the truth")
print(f"gap = 2ab*Cov = {var_sum_actual - var_sum_naive:.4f}")

**Observe:** the two expectations agree exactly (`11.5317`), while the two variances do
not — the actual variance is the larger, by the covariance term.
**Infer:** the asymmetry between the two halves of this cell is the whole point.
Expectations always add; variances only add when the components are independent, and
`chol` and `trestbps` are mildly positively correlated, so the naive formula
*understates* the spread of the composite score. Understating spread is the direction
that hurts: it produces confidence intervals that are too narrow and error bars that
are too optimistic. Session 5 measures that covariance term directly, and Session 12
shows the same term reappearing as the reason averaging many correlated models reduces
variance less than averaging independent ones would.

## Step 7 — Standardisation: putting every input on one scale

$Z = (X - \mu)/\sigma$ re-expresses a value as "how many standard deviations from the
mean", which strips the units off and makes `chol` and `thalach` directly comparable.

In [ ]:
z_scores = (df[continuous] - df[continuous].mean()) / df[continuous].std()
print("z-scores: mean and std after standardising")
print(z_scores.agg(["mean", "std"]).round(6).T)

print("\nMost extreme patient on each input:")
for col in continuous:
    i = z_scores[col].abs().idxmax()
    print(f"  {col:9} patient {i:3}  value={df.loc[i, col]:7.1f}  z={z_scores.loc[i, col]:+.2f}")

print(f"\npatients with |z| > 3 on cholesterol: {(z_scores['chol'].abs() > 3).sum()}")

**Observe:** every column now has mean `0` and std `1`, and one patient sits at
`z = +6.09` on cholesterol — a value of `564` mg/dL.
**Infer:** a z of +6 is not merely "high": under a Normal distribution it is a
one-in-a-billion observation, so seeing one in 297 patients means either the column is
not Normal (it is not — Session 3 shows it is right-skewed) or that reading is a data
error. Either way, the four patients above |z| = 3 have outsized influence on anything
that squares distances, which is most things: the variance in Step 3, the correlation
in Session 5, the least-squares fit in Session 9. Note also *what* is being
standardised. Here the whole registry is used, which is fine for description — but
doing this before a train/test split leaks test-set information into training, and
Session 10 demonstrates that failure explicitly.

## Step 8 — Skewness: is the mean even a fair summary?

Skewness measures asymmetry. Zero means symmetric; positive means a long right tail
(the mean is dragged above the median); negative means a long left tail. As a rough
rule, |skew| below 0.5 is near-symmetric, above 1 is strongly skewed.

In [ ]:
from scipy.stats import skew

print(f"{'column':10} {'skew':>7} {'mean':>9} {'median':>9}  shape")
for col in continuous:
    s = skew(df[col])
    shape = "symmetric" if abs(s) < 0.5 else ("right-skewed" if s > 0 else "left-skewed")
    print(f"{col:10} {s:+7.2f} {df[col].mean():9.1f} {df[col].median():9.1f}  {shape}")

**Observe:** `age` at `−0.22` is symmetric and `thalach` at `−0.53` just crosses
into left-skewed; `trestbps`
`+0.70`, `chol` `+1.11`, and `oldpeak` `+1.24` are right-skewed, and for each of those
the mean sits above the median.
**Infer:** the mean-above-median gap is the practical face of skew — for `chol`, mean
`247.4` versus median `243.0`, meaning "the average patient" as usually understood
(the middle one) has *less* cholesterol than the reported average. For a strongly
skewed column, quoting `mean ± std` describes a symmetric distribution that is not
there, and the median with an interquartile range is the honest summary. This directly
determines the next two sessions: `thalach` at `−0.53` is close enough to symmetric
that Session 3's Normal model and Session 7's t-test apply comfortably (a mild left
tail is not the problem a heavy right tail is), whereas `chol`
and `oldpeak` are the columns where Session 3 reaches for a log transform and Session 7
keeps a non-parametric alternative on hand.

## What this session hands to the next one

- **Expectation and variance for each input**, plus the `ddof` convention that keeps
  them consistent with the tests in Sessions 6-8.
- **A first ranking of the inputs** by how far they separate the two outcome groups
  (`thalach` and `oldpeak` strongly, `chol` barely) — refined by Session 5 and tested
  in Session 7.
- **Standardisation**, reused as a preprocessing step in Sessions 10-12, with the
  leakage caveat attached.
- **A skewness reading per column**, which is precisely the question Session 3 opens
  with: if `chol` is not symmetric, what shape *is* it, and which distribution should
  model it?

## Try it yourself

1. Recompute Step 5's group gaps using medians instead of means. Which input's ranking
   changes most, and does its skewness from Step 8 explain why?
2. Drop the four |z| > 3 cholesterol patients and re-run Steps 3 and 8. How much of
   `chol`'s skew was those four rows?
3. In Step 6, replace `trestbps` with `thalach` (which is *negatively* related to
   `chol`'s partner `age`). Does the covariance gap change sign?
4. `ca` is a count 0-3. Compute its mean and standard deviation, then argue both sides:
   when is treating it as a number defensible, and when is it not?